# Chapter 8 – Rethinking Digital Trust  
*If databases were birds, blockchain would be a duck*

This notebook explores how quantum computing is reshaping the foundations of **digital trust**. We begin with a small blockchain called **DuckChain**, adapted from *Think Blockchain*, and use it to examine three core trust pillars: **digital identity**, **hashing**, and **consensus**.

We first build a classical version of DuckChain, then strengthen its digital identities with **ML-DSA**, a lattice-based signature algorithm designed for the post-quantum era. From there, we use quantum measurement as a source of entropy for key generation. The experiment can run locally on a simulator or, with IBM Quantum access, on a physical quantum processor.

The notebook then introduces a simplified **authorization-proof** model inspired by zero-knowledge proofs. Rather than requiring named signatures for consensus, DuckChain demonstrates how approval can instead be represented by proof that an authorized participant approved a block. The teaching model keeps the mechanism visible without claiming the privacy guarantees of a production zero-knowledge system.

By the end, DuckChain combines **quantum-safe identity**, **quantum-enhanced randomness**, and **proof-based consensus**. The architecture remains familiar. The mechanisms supporting trust evolve.

### Code 8-0: Classical DuckChain

This example recreates the original DuckChain from *Think Blockchain*, adapted from its JavaScript implementation into Python. The code demonstrates the three foundational trust pillars introduced in that book: **digital identity**, **hashing**, and **consensus**. Administrators approve transactions, hashes link blocks together, and consensus determines whether a block is accepted into the ledger.

**Note:** Note: This example serves as the baseline for the chapter. The later code listings progressively enhance these same trust pillars with quantum-safe signatures, quantum measurement as a source of entropy, and proof-based authorization.

In [ ]:
# Code 8-0: DuckChain, a tiny teaching blockchain.
# Shows three core ingredients: hashing, signatures, and consensus.

import hashlib

##### Blockchain Ingredient 1: Hashing

# Create a SHA-256 fingerprint for any text input.
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Convert a block into the exact text we want to hash.
def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

# Hash a block without including its signatures.
def hash_block(block):
    return sha256(block_text(block))

##### Blockchain Ingredient 2: Signatures

# Create a toy signature for an admin approving a block hash.
def sign(admin, block_hash):
    return sha256(admin + block_hash)

# Record an admin vote by signing the proposed block hash.
def vote(admin, block):
    block["signatures"][admin] = sign(admin, block["hash"])

##### Blockchain Ingredient 3: Consensus

# Consensus means enough trusted admins signed this block.
def has_consensus(block, admins, required_signatures):
    votes = sum(
        block["signatures"].get(admin) == sign(admin, block["hash"])
        for admin in admins
    )
    return votes >= required_signatures

##### Block and chain utilities

# Create a block and hash its core contents.
def make_block(index, data, previous_hash):
    block = {
        "index": index,                  # position in the chain
        "data": data,                    # transaction or record
        "previous_hash": previous_hash,  # link to prior block
        "signatures": {},                # votes added after hashing
        "hash": ""
    }
    block["hash"] = hash_block(block)
    return block

# Verify that every block hashes correctly and links backward.
def validate_chain(chain, admins, required_signatures):
    for i in range(1, len(chain)):
        current = chain[i]
        previous = chain[i - 1]
        if current["hash"] != hash_block(current):
            return False
        if current["previous_hash"] != previous["hash"]:
            return False
        if not has_consensus(current, admins, required_signatures):
            return False
    return True

##### Demo: build a tiny DuckChain

# Step 1: Define the small network that can approve blocks.
admins = ["Huey", "Dewey", "Louie"]
required_signatures = 2

# Step 2: Start the chain with a genesis block.
duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

# Step 3: Propose a new block linked to the latest block.
transaction = "Donald pays Daisy 5 duck coins"
new_block = make_block(1, transaction, duckchain[-1]["hash"])

# Step 4: Collect enough signatures to reach consensus.
vote("Huey", new_block)
vote("Dewey", new_block)

# Step 5: Add the block only if consensus is reached.
if has_consensus(new_block, admins, required_signatures):
    duckchain.append(new_block)

# Step 6: Inspect and validate the resulting chain.
for block in duckchain:
    print(block)
print("DuckChain valid:",
      validate_chain(duckchain, admins, required_signatures))

### Code 8-1: Quantum-Safe DuckChain

This example upgrades the original DuckChain by replacing its identity layer with **ML-DSA**, a lattice-based digital signature algorithm designed to resist attacks from large-scale quantum computers. The code preserves the same three trust pillars introduced in *Think Blockchain*—digital identity, hashing, and consensus—but modernizes the identity layer using post-quantum cryptography. Administrators sign blocks using ML-DSA, consensus is established through signature verification, and the resulting ledger is validated to ensure its integrity.

**Note:** Before running this example, execute the notebook installation cell that installs the `dilithium-py` package. Later examples build on the functions defined here, so this code should be run before Codes 8-2, 8-3, and 8-4.

In [ ]:
# === Install dependency ===
!pip install dilithium-py --quiet

In [ ]:
# Code 8-1: Quantum-safe DuckChain
# Demonstrates digital identity, hashing, and consensus.

import hashlib
from dilithium_py.ml_dsa import ML_DSA_44

##### Trust Pillar 1: Digital Identity

# Create one ML-DSA keypair for each participant.
def make_admin_keys(admins):
    return {admin: ML_DSA_44.keygen() for admin in admins}

##### Trust Pillar 2: Hashing

# Create a SHA-256 fingerprint.
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Serialize the core fields of a block for hashing.
def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

# Hash a block without its signatures.
def hash_block(block):
    return sha256(block_text(block))

# Create a block linked to the previous block.
def make_block(index, data, previous_hash):
    block = {
        "index": index,                  # position in chain
        "data": data,                    # transaction payload
        "previous_hash": previous_hash,  # prior block link
        "signatures": {},                # approvals added after hashing
        "hash": ""
    }
    block["hash"] = hash_block(block)    # hash of the block's contents
    return block

##### Trust Pillar 3: Consensus

# Sign a proposed block.
def vote(admin, block, admin_keys):
    public_key, secret_key = admin_keys[admin]
    block["signatures"][admin] = ML_DSA_44.sign(
        secret_key, block["hash"].encode("utf-8")
    )

# Verify one participant's signature.
def verify_vote(admin, block, admin_keys):
    if admin not in block["signatures"]:
        return False
    public_key, _ = admin_keys[admin]
    try:
        ML_DSA_44.verify(
            public_key,
            block["hash"].encode("utf-8"),
            block["signatures"][admin]
        )
        return True
    except Exception:
        return False

# Require enough valid signatures before accepting a block.
def has_consensus(block, admins, admin_keys, required_signatures):
    votes = sum(verify_vote(admin, block, admin_keys) for admin in admins)
    return votes >= required_signatures

##### Verification

# Verify hashes, chain links, and consensus for every block.
def validate_chain(chain, admins, admin_keys, required_signatures):
    for i in range(1, len(chain)):
        current, previous = chain[i], chain[i - 1]
        if current["hash"] != hash_block(current):
            return False
        if current["previous_hash"] != previous["hash"]:
            return False
        if not has_consensus(current, admins, admin_keys, required_signatures):
            return False
    return True

##### Step 1: Establish digital identities

admins = ["Huey", "Dewey", "Louie"]
admin_keys = make_admin_keys(admins)

##### Step 2: Create and hash records

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]
transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}
new_block = make_block(1, transaction, duckchain[-1]["hash"])

##### Step 3: Reach consensus

required_signatures = 2
vote("Huey", new_block, admin_keys)
vote("Dewey", new_block, admin_keys)
if has_consensus(new_block, admins, admin_keys, required_signatures):
    duckchain.append(new_block)

##### Verify the ledger

for block in duckchain:
    print({k: v for k, v in block.items() if k != "signatures"})
    print("signatures:", list(block["signatures"].keys()))

valid = validate_chain(duckchain, admins, admin_keys, required_signatures)
print("\nDuckChain valid:", valid)

### Code 8-2: Adding Quantum Randomness to DuckChain

This example introduces the first quantum enhancement to DuckChain by using **quantum measurement outcomes** as a source of randomness for ML-DSA key generation. A single qubit is placed into superposition and measured repeatedly to produce the bits needed for each administrator’s cryptographic identity. By default, the notebook simulates those measurements locally. Readers with IBM Quantum access can switch to a physical quantum processor using the hardware setup introduced in Chapter 7.

The blockchain architecture remains unchanged. Blocks are still hashed, administrators still vote, and consensus still determines whether a block is accepted. What changes is the source of randomness used to establish those digital identities.

**Note:** Before running this example, execute the notebook installation and helper cells. Code 8-2 also assumes that Code 8-1 has already been run, since it reuses the hashing, consensus, and ledger-validation functions defined there. The helper function `measure_quantum_bits()` handles the execution details for the simulator or quantum hardware, keeping those platform-specific details separate from the experiment itself. Later examples build on the updated `make_admin_keys()` function introduced here.

In [ ]:
!pip -q install dilithium-py qiskit qiskit-aer qiskit-ibm-runtime

In [ ]:
# Code 8-2: Adding Quantum Randomness to DuckChain
# Assumes Code 8-1 and the helper cell have been run.

# Code 8-2: Quantum execution helpers

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.transpiler import generate_preset_pass_manager
from dilithium_py.ml_dsa import ML_DSA_44

USE_HARDWARE = False  # Set True to use IBM Quantum

def measure_quantum_bits(qc, shots, hardware=False):
    # Run locally unless hardware is requested.
    if not hardware:
        result = AerSimulator().run(
            qc, shots=shots, memory=True
        ).result()
        return "".join(result.get_memory())

    # Connect using the IBMQ key stored in Colab Secrets.
    from google.colab import userdata
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit_ibm_runtime.executor_sampler import Sampler

    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=userdata.get("IBMQ"),
        instance="auto"
    )
    backend = service.least_busy(
        operational=True, simulator=False
    )
    print("Quantum processor:", backend.name)

    # Adapt the circuit to the processor, then run it.
    pm = generate_preset_pass_manager(
        backend=backend, optimization_level=1
    )
    circuit = pm.run(qc)
    job = Sampler(mode=backend).run([circuit], shots=shots)
    print("Job ID:", job.job_id())

    result = job.result()
    return "".join(result[0].data.c.get_bitstrings())

##### Quantum Enhancement 1: Generate Quantum Entropy

def quantum_entropy(n_bytes, hardware=False):
    qc = QuantumCircuit(1, 1)
    qc.h(0)                   # place qubit into superposition
    qc.measure(0, 0)          # measure 0 or 1

    # Repeat measurement to collect the requested number of bits.
    bits = measure_quantum_bits(qc, n_bytes * 8, hardware)
    return int(bits, 2).to_bytes(n_bytes, "big")


##### Quantum Enhancement 2: Seed Digital Identities

def make_admin_keys(admins, hardware=False):
    global admin_seeds

    # Generate 256 bits of entropy for each administrator.
    entropy = quantum_entropy(32 * len(admins), hardware)

    admin_seeds = {
        admin: entropy[i*32:(i+1)*32]  # give each admin 32 bytes
        for i, admin in enumerate(admins)
    }

    # Derive each ML-DSA identity from its quantum seed.
    return {
        admin: ML_DSA_44.key_derive(admin_seeds[admin])
        for admin in admins
    }

##### Rebuild DuckChain

admins = ["Huey", "Dewey", "Louie"]
admin_keys = make_admin_keys(admins, USE_HARDWARE)

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}

new_block = make_block(1, transaction, duckchain[-1]["hash"])

required_signatures = 2
vote("Huey", new_block, admin_keys)
vote("Dewey", new_block, admin_keys)

if has_consensus(new_block, admins, admin_keys, required_signatures):
    duckchain.append(new_block)

valid = validate_chain(
    duckchain, admins, admin_keys, required_signatures
)

##### Inspect the Results

print("\nQuantum Randomness Experiment")
print("Source:", "IBM Quantum" if USE_HARDWARE else "Simulator")

for admin, seed in admin_seeds.items():
    bits = "".join(f"{b:08b}" for b in seed)
    print(f"\n{admin}:")
    print("  First 32 bits:", bits[:32])
    print("  Entropy sample:", seed.hex()[:32] + "...")
    print("  Zeros:", bits.count("0"), "Ones:", bits.count("1"))
    print("  Public key:", admin_keys[admin][0].hex()[:24] + "...")

print("\nDuckChain")
print("  Blocks:", len(duckchain))
print("  Signatures:", len(new_block["signatures"]))
print("  Chain valid:", valid)

### Code 8-3: Adding Zero-Knowledge Authorization to DuckChain

This example introduces the second quantum enhancement to DuckChain by replacing named signatures with **authorization proofs**. Rather than storing named signatures with a block, the ledger stores proof that an authorized participant approved it. The implementation uses a simplified hash-based construction to model the role that zero-knowledge authorization could play in the architecture.

There is an important limitation to keep in mind. Because the verifier can regenerate and compare the proof associated with each administrator, this teaching version can determine who submitted a matching proof. A real zero-knowledge system uses stronger cryptographic techniques to prove authorization without revealing that identity.

**Note:** This example assumes that Code 8-2 has already been run and reuses the hashing, ledger, and quantum-enhanced key-generation functions defined in the earlier listings. The authorization mechanism shown here models the architectural role of a zero-knowledge proof rather than providing its privacy guarantees. The complete quantum-safe, quantum-enhanced version of DuckChain is assembled in Code 8-4.

In [ ]:
# Code 8-3: Adding Zero-Knowledge Authorization to DuckChain
# Assumes Code 8-2 has already been run.

##### Quantum Enhancement 2: Authorization Proofs

# Shared credential models membership in the authorized group.
AUTHORIZED_GROUP_CREDENTIAL = "DuckChain-Admins"

def create_zk_proof(admin, block):
    # Bind authorization to this specific block.
    return sha256(
        admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"]
    )

def verify_zk_proof(proof, block, admins):
    # Regenerate each authorized proof and look for a match.
    valid_proofs = {
        sha256(admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"])
        for admin in admins
    }
    return proof in valid_proofs

def approve_block(admin, block):
    # Store proof of authorization instead of a named signature.
    block["proofs"].append(create_zk_proof(admin, block))

def has_zk_consensus(block, admins, required_proofs):
    # Consensus requires enough distinct valid proofs.
    valid_proofs = {
        proof for proof in block["proofs"]
        if verify_zk_proof(proof, block, admins)
    }
    return len(valid_proofs) >= required_proofs


##### Rebuild DuckChain with Authorization Proofs

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}
new_block = make_block(1, transaction, duckchain[-1]["hash"])
new_block["proofs"] = []  # replace signatures with authorization proofs

approve_block("Huey", new_block)
approve_block("Dewey", new_block)

required_proofs = 2
if has_zk_consensus(new_block, admins, required_proofs):
    duckchain.append(new_block)


##### Display Results

print("\nAuthorization proofs:")
for proof in new_block["proofs"]:
    print(" ", proof[:16] + "...")

print("Proof count:", len(new_block["proofs"]))
print("Consensus reached:",
      has_zk_consensus(new_block, admins, required_proofs))

### Code 8-4: DuckChain, Quantum-Safe and Quantum-Enhanced

This example assembles the chapter's enhancements into a single reference implementation of DuckChain. The ledger combines **ML-DSA quantum-safe identities**, **quantum entropy**, and **proof-based consensus** with the hashing and linked blocks already familiar from DuckChain. The architecture still rests on the same three trust pillars: digital identity, integrity, and consensus. What changes is how those pillars are implemented for a quantum era.

The complete version also carries forward the simulator/hardware option from Code 8-2. With `USE_HARDWARE = False`, quantum measurements are simulated locally. Set it to `True`, and the same entropy-generating circuit runs on an IBM Quantum processor. The execution details remain tucked inside `measure_quantum_bits()`, leaving the DuckChain implementation unchanged.

**Note:** Code 8-4 builds on the earlier examples and serves as the chapter's consolidated DuckChain implementation. Run the notebook setup cells and Codes 8-1 through 8-3 first. In particular, Code 8-4 reuses the quantum execution helper introduced with Code 8-2. The authorization proofs remain a simplified teaching model of the role zero-knowledge proofs could play; they do not provide the privacy guarantees of a production zero-knowledge system.

In [ ]:
# Code 8-4: DuckChain, Quantum-Safe and Quantum-Enhanced
# Assumes the Code 8-2 quantum execution helper has been run.

import hashlib
from qiskit import QuantumCircuit
from dilithium_py.ml_dsa import ML_DSA_44

USE_HARDWARE = False  # Set True to use IBM Quantum


##### Quantum Enhancement 1: Generate Quantum Entropy

def quantum_entropy(n_bytes, hardware=False):
    qc = QuantumCircuit(1, 1)
    qc.h(0)                   # place qubit into superposition
    qc.measure(0, 0)          # measure 0 or 1

    bits = measure_quantum_bits(qc, n_bytes * 8, hardware)
    return int(bits, 2).to_bytes(n_bytes, "big")


##### Trust Pillar 1: Quantum-Safe Digital Identity

def make_admin_keys(admins, hardware=False):
    entropy = quantum_entropy(32 * len(admins), hardware)

    seeds = {
        admin: entropy[i*32:(i+1)*32]
        for i, admin in enumerate(admins)
    }
    return {
        admin: ML_DSA_44.key_derive(seeds[admin])
        for admin in admins
    }


##### Trust Pillar 2: Hashing

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

def hash_block(block):
    return sha256(block_text(block))

def make_block(index, data, previous_hash):
    block = {
        "index": index,
        "data": data,
        "previous_hash": previous_hash,
        "proofs": [],
        "hash": ""
    }
    block["hash"] = hash_block(block)
    return block


##### Quantum Enhancement 2: Authorization Proofs

AUTHORIZED_GROUP_CREDENTIAL = "DuckChain-Admins"

def create_zk_proof(admin, block):
    return sha256(
        admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"]
    )

def verify_zk_proof(proof, block, admins):
    valid_proofs = {
        sha256(admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"])
        for admin in admins
    }
    return proof in valid_proofs

def approve_block(admin, block):
    block["proofs"].append(create_zk_proof(admin, block))


##### Trust Pillar 3: Proof-Based Consensus

def has_zk_consensus(block, admins, required_proofs):
    valid_proofs = {
        proof for proof in block["proofs"]
        if verify_zk_proof(proof, block, admins)
    }
    return len(valid_proofs) >= required_proofs


##### Verification

def validate_zk_chain(chain, admins, required_proofs):
    for i in range(1, len(chain)):
        current, previous = chain[i], chain[i - 1]

        if current["hash"] != hash_block(current):
            return False                 # block contents changed
        if current["previous_hash"] != previous["hash"]:
            return False                 # chain link broken
        if not has_zk_consensus(
            current, admins, required_proofs
        ):
            return False                 # consensus missing

    return True

##### Build DuckChain

admins = ["Huey", "Dewey", "Louie"]

# Establish quantum-seeded ML-DSA identities.
admin_keys = make_admin_keys(admins, USE_HARDWARE)

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}
new_block = make_block(1, transaction, duckchain[-1]["hash"])

required_proofs = 2
approve_block("Huey", new_block)
approve_block("Dewey", new_block)

if has_zk_consensus(new_block, admins, required_proofs):
    duckchain.append(new_block)


##### Display Results

source = "IBM Quantum" if USE_HARDWARE else "Simulator"

print("\nComplete DuckChain")
print("Randomness source:", source)
print("Blocks:", len(duckchain))
print("Authorization proofs:", len(new_block["proofs"]))
print("Consensus reached:",
      has_zk_consensus(new_block, admins, required_proofs))
print("Chain valid:",
      validate_zk_chain(duckchain, admins, required_proofs))

print("\nLatest block")
print("Data:", new_block["data"])
print("Previous hash:", new_block["previous_hash"][:16] + "...")
print("Block hash:", new_block["hash"][:16] + "...")